# Policy Verification Agent — Multi-Tool MCP

The **Policy Verification Agent** is the third specialist in the Phase 1 pipeline. After the Authenticator confirms identity and the Extractor pulls structured data from every claim document, the Policy Verification Agent applies three verification checks against the Socotra policy administration system:

1. **Coverage status** — is the policy active and premiums current?
2. **Death benefit calculation** — what is the payable benefit amount?
3. **Exclusion check** — do the death circumstances trigger any policy exclusions?

This agent uses the same Socotra MCP server introduced in notebook `01_single_tool_mcp`, extended with the three tools needed for policy verification. Unlike the Extractor Agent (which used a single `@tool`), this agent calls **three MCP tools in sequence**, reasoning over the combined results before producing its output.

### What's New in This Notebook

| Concept | Description |
|---|---|
| Multi-tool MCP agent | Agent selects and sequences three MCP tools based on reasoning |
| Claude Sonnet 4 | Swapping models for complex policy reasoning — architecture-specified |
| Structured verification decision | Agent produces a typed output with per-check results and an overall recommendation |
| Cross-document field matching | Agent compares extracted fields across documents to detect inconsistencies |

### How It Fits in the Pipeline

```
Authenticator Agent  →  Extractor Agent  →  Policy Verification Agent  →  Supervisor / Adjudicator
   (notebook 01-02)              (notebook 02)                (this notebook)              (future)
```

The Extractor Agent produces JSON extraction files for each claim document (produced in the current notebook directory). The Policy Verification Agent loads these extractions, identifies the policy number and death circumstances, then calls three Socotra MCP tools to verify the claim against the policy administration system.

## Setup

In [ ]:
import boto3
from botocore.exceptions import ClientError
import json
import sys
import os

import re
from decimal import Decimal
from datetime import datetime, timezone


from pathlib import Path
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from mcp import stdio_client, StdioServerParameters

# ── Path setup ────────────────────────────────────────────────────────────────
# Notebooks:   insurance-claims-processing/notebooks/02_integrate_agent_tools/
# MCP servers: insurance-claims-processing/mcp_servers/socotra_mock/
# Sample data: insurance-claims-processing/sample_data/
REPO_ROOT        = Path("../..").resolve()
MCP_SERVER_PATH  = REPO_ROOT / "mcp_servers" / "socotra_mock"
MOCK_DATA_PATH   = MCP_SERVER_PATH / "mock_data.json"
SAMPLE_DATA_PATH = REPO_ROOT / "sample_data"

# ── Model configuration ───────────────────────────────────────────────────────
# Claude 4 Sonnet for complex policy reasoning (architecture-specified)
MODEL_ID = "us.anthropic.claude-sonnet-4-20250514-v1:0"
REGION = boto3.session.Session().region_name or "us-east-1"

model = BedrockModel(
    model_id=MODEL_ID,
    region_name=REGION,
    max_tokens=4096,
    temperature=0.1,
)

print(f"✅ Claude Sonnet 4 configured for policy verification")
print(f"   Model           : {MODEL_ID}")
print(f"   MCP server path : {MCP_SERVER_PATH}")
print(f"   Sample data     : {SAMPLE_DATA_PATH}")

## Part A — Load Extractor Output

The Policy Verification Agent consumes the JSON files produced by the Extractor Agent in notebook `02_document_processing_agent`. These files are produced in the current notebook directory and represent the structured extraction payload that would flow through the pipeline in production.

We load all seven extracted documents and build a consolidated payload — the same structure the Supervisor Agent would pass to the Policy Verification Agent.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Load all extracted JSON files from current directory
# These are the type of outputs of the Extractor Agent (notebook 02)
# ─────────────────────────────────────────────────────────────────────────────

EXTRACTED_FILES = {
    "death_certificate": "death_certificate_sample_extracted.json",
    "policy_document":   "policy_whole_life_sample_extracted.json",
    "medical_records":   "medical_records_sample_extracted.json",
    "will_document":     "will_document_sample_extracted.json",
    "trust_document":    "trust_document_sample_extracted.json",
    "beneficiary_id":    "beneficiary_id_sample_extracted.json",
    "police_report":     "police_report_sample_extracted.json",
}

extraction_payload = {}

print("📂 Loading Extractor Agent output from current directory")
print()

for doc_type, filename in EXTRACTED_FILES.items():
    filepath = Path.cwd() / filename
    if filepath.exists():
        with open(filepath) as f:
            extraction_payload[doc_type] = json.load(f)
        print(f"   ✅ {doc_type:<25} ← {filename}")
    else:
        print(f"   ❌ {doc_type:<25} ← {filename} (NOT FOUND)")
        extraction_payload[doc_type] = {"error": "File not found"}

print()
print(f"   Documents loaded: {len(extraction_payload)}")

### Identify Key Fields for Verification

Before calling the MCP tools, we extract the key fields the agent needs from the extraction payload. This mirrors what the agent will do internally — pull the policy number from the policy document, the death circumstances from the death certificate and police report, and the date of death for benefit calculation.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Extract key fields the Policy Verification Agent needs
# These come from the Extractor Agent's structured output
# ─────────────────────────────────────────────────────────────────────────────

# Policy number from the extracted policy document
policy_number = extraction_payload.get("policy_document", {}).get(
    "policy_overview", {}).get("policy_number", "UNKNOWN")

# Death date from the death certificate
death_date = extraction_payload.get("death_certificate", {}).get(
    "death_information", {}).get("date_of_death", "UNKNOWN")

# Death circumstances — combine death certificate manner + cause + police narrative
death_manner = extraction_payload.get("death_certificate", {}).get(
    "death_information", {}).get("manner", "")
immediate_cause = extraction_payload.get("death_certificate", {}).get(
    "cause_of_death", {}).get("immediate_cause_a", "")
police_narrative = extraction_payload.get("police_report", {}).get(
    "narrative", "")

death_circumstances = f"{death_manner} — {immediate_cause}. Police: {police_narrative}"

# Insured name from policy
insured_name = extraction_payload.get("policy_document", {}).get(
    "insured_owner", {}).get("name", "UNKNOWN")

# Beneficiary from policy
beneficiary_name = extraction_payload.get("policy_document", {}).get(
    "beneficiary_designation", {}).get("primary_beneficiary", "UNKNOWN")

print("🔑 Key fields extracted from Extractor output:")
print(f"   Policy number     : {policy_number}")
print(f"   Insured           : {insured_name}")
print(f"   Beneficiary       : {beneficiary_name}")
print(f"   Date of death     : {death_date}")
print(f"   Manner of death   : {death_manner}")
print(f"   Immediate cause   : {immediate_cause}")
print(f"   Police narrative  : {police_narrative[:100]}...")

## Part B — Connect to the Socotra MCP Server

We connect to the same Socotra mock MCP server used in `01_single_tool_mcp`. The server exposes five tools — the Policy Verification Agent will use three of them:

| MCP Tool | Verification Check | What It Returns |
|---|---|---|
| `verify_coverage_status` | Coverage status | Active/lapsed flag, premium status, policy type, dates |
| `calculate_death_benefit` | Death benefit | Payable benefit amount with breakdown |
| `check_exclusions` | Exclusion check | Triggered exclusions, eligibility flag |

The agent discovers these tools at runtime via MCP — it doesn't need to know their implementation.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Configure the Socotra MCP client — same server as notebook 01
# The Policy Verification Agent uses 3 of the 5 available tools:
#   verify_coverage_status, calculate_death_benefit, check_exclusions
# ─────────────────────────────────────────────────────────────────────────────

SOCOTRA_SERVER_SCRIPT = str(MCP_SERVER_PATH / "server.py")

socotra_mcp_client = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(
            command=sys.executable,
            args=[SOCOTRA_SERVER_SCRIPT],
            env={
                "MOCK_DATA_PATH": str(MOCK_DATA_PATH),
                **os.environ
            }
        )
    )
)

print("✅ MCP client configured")
print(f"   Python : {sys.executable}")
print(f"   Server : {SOCOTRA_SERVER_SCRIPT}")

## Part C — Define the Policy Verification Agent

The system prompt instructs the agent to:
1. Call three MCP tools in sequence — coverage, benefit, exclusions
2. Cross-reference extracted document fields to detect inconsistencies (e.g., decedent name on death certificate vs. insured name on policy)
3. Produce a structured verification decision with per-check results and an overall recommendation

This is the key architectural shift from the Authenticator (which called tools to fetch data for identity checks) to the Policy Verification Agent (which calls tools to verify policy terms and produces a coverage decision).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Policy Verification Agent system prompt
#
# Key differences from the Authenticator:
#   - Uses 3 MCP tools (coverage, benefit, exclusions) not 4
#   - Cross-references fields across extracted documents
#   - Produces a structured verification decision, not an auth status
# ─────────────────────────────────────────────────────────────────────────────

POLICY_VERIFICATION_SYSTEM_PROMPT = """You are the Policy Verification Agent for an Insurance Claims Processing workflow.

Your role is to verify policy terms against the Socotra policy administration system
and produce a structured verification decision for the downstream Adjudicator.

## Input

You receive a JSON payload containing structured extractions from all claim documents
(produced by the Extractor Agent). Key documents:
- policy_document: policy number, face amount, beneficiary designation
- death_certificate: date of death, cause/manner of death, decedent identity
- police_report: death investigation narrative, disposition
- medical_records: patient conditions, treatment history
- beneficiary_id: claimant identity document
- will_document: estate disposition, insurance proceeds instructions
- trust_document: trust beneficiaries, trustee designations

## Your Verification Workflow

Use your MCP tools in this sequence:

1. **verify_coverage_status(policy_number)**
   - Confirm the policy is active and premiums are current
   - Compare the policy type and issue date against the extracted policy document
   - If lapsed: note the lapse date and reason

2. **calculate_death_benefit(policy_number, death_date)**
   - Get the payable benefit amount from Socotra
   - Compare against the face amount in the extracted policy document
   - Flag any discrepancy between Socotra's calculation and the document

3. **check_exclusions(policy_number, death_circumstances)**
   - Pass the death circumstances (manner + cause + police narrative)
   - Check whether any policy exclusions are triggered
   - Cross-reference with medical records for consistency

## Cross-Document Consistency Checks

After all three tool calls, perform these field-matching checks:
- Decedent name on death certificate vs. insured name on policy
- Date of birth on death certificate vs. policy vs. medical records
- Address on death certificate vs. policy
- Beneficiary name on policy vs. beneficiary ID document
- Policy number referenced in will/trust vs. extracted policy number
- Cause of death on certificate vs. medical history (plausibility)

## Output Format

Return your verification decision as a structured JSON object:

```json
{
  "verification_decision": {
    "policy_number": "<policy_number>",
    "overall_status": "VERIFIED | FLAGGED | DENIED",
    "recommended_action": "Proceed to Adjudication | Escalate for Manual Review | Deny Claim",
    "coverage_check": {
      "status": "PASS | FAIL",
      "policy_active": true/false,
      "premium_status": "<status>",
      "policy_type": "<type>",
      "notes": "<details>"
    },
    "benefit_check": {
      "status": "PASS | FLAGGED",
      "socotra_benefit_amount": <amount>,
      "document_face_amount": <amount>,
      "amounts_match": true/false,
      "notes": "<details>"
    },
    "exclusion_check": {
      "status": "CLEAR | TRIGGERED",
      "exclusions_triggered": [],
      "claim_eligible": true/false,
      "notes": "<details>"
    },
    "cross_document_consistency": {
      "status": "CONSISTENT | INCONSISTENCIES_FOUND",
      "checks": [
        {"field": "<field_name>", "status": "MATCH | MISMATCH", "details": "<details>"}
      ]
    },
    "flags": ["<any anomalies or concerns for the adjudicator>"]
  }
}
```

## Guidelines
- Always run all three tool calls — do not short-circuit on a single failure
- You verify policy terms — you do NOT make final payout decisions
- Any triggered exclusion requires escalation, not auto-denial
- Cross-document mismatches must be flagged, not silently ignored
- Protect PII — do not repeat SSNs or full DOBs unnecessarily
"""

print(f"✅ Policy Verification system prompt defined ({len(POLICY_VERIFICATION_SYSTEM_PROMPT)} chars)")
print("   Verification checks: coverage status, death benefit, exclusions")
print("   Cross-document checks: name, DOB, address, beneficiary, policy ref, cause of death")

### Build the Verification Request

We assemble the verification request from the Extractor output. In production, the Supervisor Agent would construct this payload and route it to the Policy Verification Agent. Here we build it manually from the loaded extraction files.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Build the verification request payload
# In production: Supervisor Agent constructs this from Extractor output
# Here: we build it from the loaded JSON files
# ─────────────────────────────────────────────────────────────────────────────

verification_request_payload = json.dumps(extraction_payload, indent=2)

VERIFICATION_REQUEST = f"""## Policy Verification Request

Please verify the following claim against the Socotra policy administration system.

**Policy Number:** {policy_number}
**Insured:** {insured_name}
**Beneficiary:** {beneficiary_name}
**Date of Death:** {death_date}
**Death Circumstances:** {death_manner} — {immediate_cause}

### Extracted Document Payload

The following JSON contains structured extractions from all claim documents
(produced by the Extractor Agent):

```json
{verification_request_payload}
```

Please run all three verification checks (coverage, benefit, exclusions) using
your MCP tools, then perform cross-document consistency checks and return your
structured verification decision.
"""

print(f"✅ Verification request built ({len(VERIFICATION_REQUEST)} chars)")
print(f"   Policy  : {policy_number}")
print(f"   Insured : {insured_name}")
print(f"   Death   : {death_date} — {death_manner}")

## Part D — Run the Policy Verification Agent

Now we connect to the MCP server, discover the available tools, build the agent, and run the verification. Watch the output — the agent will call three MCP tools in sequence, then reason over the combined results to produce its verification decision.

The `with socotra_mcp_client:` context manager starts the MCP server process, and the agent discovers tools at runtime via `list_tools_sync()`.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Run the Policy Verification Agent with MCP tools
# The agent will call: verify_coverage_status, calculate_death_benefit,
# check_exclusions — then cross-reference extracted document fields
# ─────────────────────────────────────────────────────────────────────────────

with socotra_mcp_client:
    # Discover available tools from the running MCP server
    mcp_tools = socotra_mcp_client.list_tools_sync()

    print("🔧 Tools discovered from Socotra MCP server:")
    for t in mcp_tools:
        print(f"   - {t.tool_name}")
    print()

    # Build the Policy Verification Agent
    policy_verification_agent = Agent(
        model=model,
        system_prompt=POLICY_VERIFICATION_SYSTEM_PROMPT,
        tools=mcp_tools,
    )

    print("🤖 Running Policy Verification Agent...")
    print("=" * 60)
    print("💡 Watch: the agent calls 3 MCP tools, then reasons over combined results")
    print()

    verification_response = policy_verification_agent(VERIFICATION_REQUEST)

    print()
    print("=" * 60)
    print("✅ Policy verification complete")

### Inspect the Reasoning + Tool Call Trace

The trace shows the agent's multi-tool reasoning loop:
1. **Reasoning** — the agent plans which tools to call and in what order
2. **Tool calls** — three MCP tool invocations in sequence (coverage → benefit → exclusions)
3. **Cross-document analysis** — the agent compares fields across extracted documents
4. **Verification decision** — structured JSON output with per-check results

This is the key difference from the Authenticator: the agent sequences multiple tools and synthesizes their results into a single decision.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Parse and display the agent's reasoning + tool call trace
# ─────────────────────────────────────────────────────────────────────────────

print("🔍 Full Agent Trace")
print()

for i, message in enumerate(policy_verification_agent.messages):
    role    = message.get("role", "unknown")
    content = message.get("content", [])

    if isinstance(content, str):
        blocks = [{"type": "text", "text": content}]
    elif isinstance(content, list):
        blocks = content
    else:
        blocks = [{"type": "text", "text": str(content)}]

    print(f"── Message {i} | role={role} ──────────────────────────────")

    for block in blocks:
        if not isinstance(block, dict):
            print(f"   [raw block] {str(block)[:200]}")
            continue

        block_type = block.get("type", "unknown")

        if block_type == "reasoningContent":
            reasoning_raw = block.get("reasoningText", "")
            reasoning_text = reasoning_raw.get("text", str(reasoning_raw)) if isinstance(reasoning_raw, dict) else str(reasoning_raw)
            print(f"🧠 REASONING ({len(reasoning_text)} chars):")
            print(reasoning_text[:500])
            if len(reasoning_text) > 500:
                print(f"   ... ({len(reasoning_text) - 500} more chars)")
            print()

        elif block_type == "tool_use":
            print(f"🔧 TOOL CALL : {block.get('name', 'unknown')}")
            print(f"   Tool ID   : {block.get('id', 'n/a')}")
            print(f"   Input     : {json.dumps(block.get('input', {}), indent=2)[:300]}")
            print()

        elif block_type == "tool_result":
            result_content = block.get("content", "")
            if isinstance(result_content, list):
                for rc in result_content:
                    if isinstance(rc, dict) and rc.get("type") == "text":
                        result_preview = rc.get("text", "")[:300]
                    else:
                        result_preview = str(rc)[:300]
            else:
                result_preview = str(result_content)[:300]
            print(f"   ↳ Tool result: {result_preview}")
            print()

        elif block_type == "text":
            text = block.get("text", "")
            if text.strip():
                print(f"📋 TEXT ({role}):")
                print(text[:2000])
                if len(text) > 2000:
                    print(f"   ... ({len(text) - 2000} more chars)")
                print()

        else:
            print(f"   [block type={block_type}] {str(block)[:200]}")
            print()

## Part E — Edge Case: Lapsed Policy

Now test with a lapsed policy (`POL-WL-2023-042`). The agent should detect the lapsed status in the coverage check and flag the claim accordingly. This tests the agent's ability to handle a negative verification result and still complete all three checks.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Edge case: lapsed policy
# The agent should detect lapsed status and flag the claim
# ─────────────────────────────────────────────────────────────────────────────

LAPSED_POLICY_REQUEST = """## Policy Verification Request

Please verify the following claim against the Socotra policy administration system.

**Policy Number:** POL-WL-2023-042
**Insured:** Robert James Wilson
**Beneficiary:** Emily Wilson
**Date of Death:** 2025-02-01
**Death Circumstances:** Natural causes

### Extracted Document Payload

```json
{
  "policy_document": {
    "policy_overview": {
      "policy_number": "POL-WL-2023-042",
      "plan": "Whole Life Insurance",
      "issue_date": "2019-06-10",
      "status": "Lapsed",
      "face_amount": 250000.00,
      "premium_mode": "Annual",
      "annual_premium": 4200.00
    },
    "insured_owner": {
      "name": "Robert James Wilson",
      "date_of_birth": "1970-11-05",
      "address": "456 Oak Ave, Portland, OR 97201"
    },
    "beneficiary_designation": {
      "primary_beneficiary": "Emily Wilson",
      "relationship": "daughter",
      "benefit_percent": 100
    },
    "policy_values": {
      "cash_surrender_value": 18500.00,
      "outstanding_policy_loan": 0.00,
      "net_death_benefit_estimated": 0.00
    }
  },
  "death_certificate": {
    "decedent_information": {
      "name_last_first": "Wilson, Robert James",
      "dob": "1970-11-05"
    },
    "death_information": {
      "date_of_death": "2025-02-01",
      "manner": "Natural"
    },
    "cause_of_death": {
      "immediate_cause_a": "Cardiac arrest"
    }
  }
}
```

Please run all three verification checks and return your structured verification decision.
"""

with socotra_mcp_client:
    mcp_tools = socotra_mcp_client.list_tools_sync()

    lapsed_agent = Agent(
        model=model,
        system_prompt=POLICY_VERIFICATION_SYSTEM_PROMPT,
        tools=mcp_tools,
    )

    print("🔴 Testing lapsed policy scenario (POL-WL-2023-042)...")
    print("=" * 60)
    print()

    lapsed_response = lapsed_agent(LAPSED_POLICY_REQUEST)

    print()
    print("=" * 60)
    print("✅ Lapsed policy verification complete")

## Part F — Edge Case: Exclusion Triggered

Test with death circumstances that trigger a policy exclusion. The agent should detect the exclusion in the check_exclusions tool result and flag the claim for manual review. This tests the agent's ability to handle exclusion triggers without auto-denying the claim.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Edge case: lapsed policy
# The agent should detect lapsed status and flag the claim
# ─────────────────────────────────────────────────────────────────────────────

LAPSED_POLICY_REQUEST = """## Policy Verification Request

Please verify the following claim against the Socotra policy administration system.

**Policy Number:** POL-WL-2023-042
**Insured:** Robert James Wilson
**Beneficiary:** Emily Wilson
**Date of Death:** 2025-02-01
**Death Circumstances:** Natural causes

### Extracted Document Payload

```json
{
  "policy_document": {
    "policy_overview": {
      "policy_number": "POL-WL-2023-042",
      "plan": "Whole Life Insurance",
      "issue_date": "2019-06-10",
      "status": "Lapsed",
      "face_amount": 250000.00,
      "premium_mode": "Annual",
      "annual_premium": 4200.00
    },
    "insured_owner": {
      "name": "Robert James Wilson",
      "date_of_birth": "1970-11-05",
      "address": "456 Oak Ave, Portland, OR 97201"
    },
    "beneficiary_designation": {
      "primary_beneficiary": "Emily Wilson",
      "relationship": "daughter",
      "benefit_percent": 100
    },
    "policy_values": {
      "cash_surrender_value": 18500.00,
      "outstanding_policy_loan": 0.00,
      "net_death_benefit_estimated": 0.00
    }
  },
  "death_certificate": {
    "decedent_information": {
      "name_last_first": "Wilson, Robert James",
      "dob": "1970-11-05"
    },
    "death_information": {
      "date_of_death": "2025-02-01",
      "manner": "Natural"
    },
    "cause_of_death": {
      "immediate_cause_a": "Cardiac arrest"
    }
  }
}
```

Please run all three verification checks and return your structured verification decision.
"""

with socotra_mcp_client:
    mcp_tools = socotra_mcp_client.list_tools_sync()

    lapsed_agent = Agent(
        model=model,
        system_prompt=POLICY_VERIFICATION_SYSTEM_PROMPT,
        tools=mcp_tools,
    )

    print("🔴 Testing lapsed policy scenario (POL-WL-2023-042)...")
    print("=" * 60)
    print()

    lapsed_response = lapsed_agent(LAPSED_POLICY_REQUEST)

    print()
    print("=" * 60)
    print("✅ Lapsed policy verification complete")

## Part G — Store Verification Decision in DynamoDB

The Policy Verification Agent writes its verification decision to **Amazon DynamoDB** as initial claim metadata. This record is consumed by downstream agents.

In this section we:
1. Create a DynamoDB table for claim metadata 
2. Write the verification decision from Part D as a claim record
3. Read it back to confirm persistence

### Create the Claims Metadata Table

We create a DynamoDB table with `claim_id` as the partition key. 

In [ ]:
dynamodb = boto3.resource("dynamodb", region_name=REGION)
TABLE_NAME = "claims-metadata"

try:
    table = dynamodb.create_table(
        TableName=TABLE_NAME,
        KeySchema=[
            {"AttributeName": "claim_id", "KeyType": "HASH"}
        ],
        AttributeDefinitions=[
            {"AttributeName": "claim_id", "AttributeType": "S"}
        ],
        BillingMode="PAY_PER_REQUEST"
    )
    table.wait_until_exists()
    print(f"✅ Created DynamoDB table: {TABLE_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] == "ResourceInUseException":
        table = dynamodb.Table(TABLE_NAME)
        print(f"✅ DynamoDB table already exists: {TABLE_NAME}")
    else:
        raise

print(f"   Table status: {table.table_status}")


### Write the Verification Decision

We parse the agent's response to extract the structured verification decision JSON, then write it to DynamoDB as a claim record. The `claim_id` is derived from the policy number and date of death.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Write verification decision to DynamoDB
# ─────────────────────────────────────────────────────────────────────────────


def float_to_decimal(obj):
    """Convert floats to Decimal for DynamoDB compatibility."""
    if isinstance(obj, float):
        return Decimal(str(obj))
    elif isinstance(obj, dict):
        return {k: float_to_decimal(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [float_to_decimal(i) for i in obj]
    return obj

# Extract the JSON verification decision from the agent response
response_text = str(verification_response)
json_match = re.search(r"```json\s*(\{[\s\S]+?\})\s*```", response_text)

if not json_match:
    # Try without code fences
    json_match = re.search(r'(\{[\s\S]*verification_decision[\s\S]*\})', response_text)

if json_match:
    verification_decision = json.loads(json_match.group(1))
    print("✅ Parsed verification decision from agent response")
else:
    # Fallback: store the raw response text
    verification_decision = {"raw_response": response_text[:5000]}
    print("⚠️  Could not parse structured JSON — storing raw response")

# Build the claim record
claim_id = f"CLM-{policy_number}-{death_date}"

claim_record = {
    "claim_id": claim_id,
    "policy_number": policy_number,
    "insured_name": insured_name,
    "beneficiary_name": beneficiary_name,
    "date_of_death": death_date,
    "verification_decision": verification_decision,
    "agent_model": MODEL_ID,
    "created_at": datetime.now(timezone.utc).isoformat() + "Z",
    "stage": "policy_verification_complete"
}

# Convert floats to Decimal (DynamoDB requirement)
claim_record_ddb = float_to_decimal(claim_record)

table.put_item(Item=claim_record_ddb)

print(f"✅ Claim record written to DynamoDB")
print(f"   Table    : {TABLE_NAME}")
print(f"   Claim ID : {claim_id}")
print(f"   Stage    : policy_verification_complete")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Test: Read back the claim record from DynamoDB
# ─────────────────────────────────────────────────────────────────────────────

response_ddb = table.get_item(Key={"claim_id": claim_id})
item = response_ddb.get("Item", {})

if item:
    print(f"✅ Claim record retrieved from DynamoDB")
    print(f"   Claim ID : {item['claim_id']}")
    print(f"   Policy   : {item['policy_number']}")
    print(f"   Stage    : {item['stage']}")
    print(f"   Created  : {item['created_at']}")
    print()
    print("   Verification decision (first 500 chars):")
    print(f"   {json.dumps(item.get('verification_decision', {}), indent=2, default=str)[:500]}")
else:
    print(f"❌ Claim record not found for {claim_id}")


## ✅ Notebook Complete: Policy Verification Agent

### What You Built

- **Policy Verification Agent** that calls three MCP tools in sequence to verify a claim
- Cross-document field matching to detect inconsistencies across extracted documents
- Structured verification decision output for downstream consumption

### Key Patterns Used

| Pattern | Description |
|---|---|
| **Multi-tool MCP agent** | Agent discovers and sequences three MCP tools based on its reasoning — coverage, benefit, exclusions. Tool selection is driven by the system prompt, not hardcoded. |
| **Cross-document validation** | Agent compares fields across independently extracted documents (death cert vs. policy vs. medical records) to detect inconsistencies that a single-document check would miss. |
| **Structured decision output** | Agent produces a typed JSON object with per-check results and an overall recommendation — enabling downstream agents to consume the decision programmatically. |
| **Multi-model agentic worfklow** | Claude Sonnet 4 replaces Nova 2 Lite for for multi-step policy verification. |

### Limitations of This Notebook

| Limitation | Why It Exists | What Addresses It |
|---|---|---|
| No claim state persistence | Verification results exist only in memory | Supervisor Agent writes to DynamoDB |
| No S3 integration | Extraction payload loaded from local files | Supervisor Agent owns S3 retrieval |
| No automated handoff | Verification decision printed, not routed | Multi-agent orchestration (notebook 03) |
| Mock policy data only | Socotra MCP server uses static JSON | Production: live Socotra API integration |